In [1]:
import pandas as pd
import re
import json

In [2]:
# Define the path to the CSV you want to load:
# path = '/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/final/nacc/train_with_summary/qwen25_72B_etpr_cog_mci_park.csv'
# path = '/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/final/nacc/train_with_summary/qwen3_14B_etpr_cog_mci_park_nocop.csv'
# path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/fm_adrd/adrd_simplified_evaluation/benchmarks/adrd_cog_status_summary/COG_STATUS_test.csv"
# path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/fm_adrd/adrd_simplified_evaluation/benchmarks/adrd_neuropath_summary/NP_MAJOR_test.csv"
# path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/train_summary.csv"
path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/train_summary.csv"

# Read the data
summaries = pd.read_csv(path)


In [3]:
# x = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/train_summary_32B.csv")

In [4]:
# df1 = summaries.sort_index().sort_index(axis=1)
# df2 = x.sort_index().sort_index(axis=1)

# # Step 2: Create a boolean mask of differences
# diff_mask = df1 != df2

# # Step 3: Get the actual differences (optional but useful for debugging)
# diff_df = df1[diff_mask]  # Shows differing values from df1
# diff_df2 = df2[diff_mask] # Shows differing values from df2

# # Step 4: Find the specific row/column indices of the differences
# differences = diff_mask.stack()
# differences = differences[differences]  # Keep only True values

# # Print row/column locations
# print("Differences found at:")
# print(differences.index.tolist())

In [17]:
# dp = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/dropped_rerun.csv")
# for i, row in dp.iterrows():
#     summaries.loc[summaries["ID"] == row["ID"], "visit_summary"] = row["visit_summary"]

In [3]:
summaries.head()

,ID,patient_summary,diag_summary,question,options,ground_truth,Q_TYPE,visit_summary
0,NACC529204,"{\n ""Subject Demographics"": {\n ""Liv...","{\n ""Clinician Judgment of Symptoms"": {\n ...","Based on the clinical data, identify the neuro...",A. Alzheimer's disease neuropathology (AD)\nB....,H,Neuropath,The subject is a 100-year-old right-handed Whi...
1,NACC834497,"{\n ""Subject Demographics"": {\n ""Liv...","{\n ""Clinician Judgment of Symptoms"": {\n ...","Based on the clinical data, identify the neuro...",A. Alzheimer's disease neuropathology (AD)\nB....,C,Neuropath,The patient is an 88-year-old male who resides...
2,NACC029705,"{\n ""Subject Demographics"": {\n ""Liv...","{\n ""Clinician Judgment of Symptoms"": {\n ...","Based on the details provided, identify if the...",A. Yes\nB. No,B,Amyloid PET,"The patient is a 76-year-old left-handed, marr..."
3,NACC396023,"{\n ""Subject Demographics"": {\n ""Liv...","{\n ""Clinician Judgment of Symptoms"": {\n ...","Based on the patient's data, identify the appr...",A. Amnestic MCI- single domain\nB. Amnestic MC...,E,MCI subtype,The patient is an 81-year-old right-handed mal...
4,NACC362586,"{\n ""Subject Demographics"": {\n ""Liv...","{\n ""Clinician Judgment of Symptoms"": {\n ...","From the information available, determine the ...",A. Normal Cognition (NC)\nB. Mild Cognitive Im...,A,Cognitive status,"The patient is a 69-year-old left-handed, Engl..."


In [4]:
len(summaries)

45041

In [5]:
import pandas as pd
import re
from collections import Counter

def find_repeated_sentences_in_text(text):
    # Split into sentences using punctuation (adjust if needed)
    sentences = re.split(r'(?<=[.!?])\s+', str(text).strip())
    normalized = [s.strip().lower() for s in sentences if s.strip()]
    counter = Counter(normalized)
    repeated = {sent: count for sent, count in counter.items() if count > 1}
    return repeated

# Load your dataframe
# df = pd.read_csv("your_file.csv")  # Example
# Example dataframe:
# df = pd.DataFrame({'visit_summary': [...texts...]})

repeats_info = []

for idx, row in summaries.iterrows():
    text = row['visit_summary']
    repeats = find_repeated_sentences_in_text(text)
    if repeats:
        for sentence, count in repeats.items():
            repeats_info.append({
                "row_index": idx,
                "sentence": sentence,
                "count": count
            })

# Convert results to a DataFrame for easy analysis
repeats_df = pd.DataFrame(repeats_info)

In [6]:
if 'row_index' in repeats_df:
    repeat_list = set(repeats_df['row_index'])
else:
    repeat_list = set()

In [7]:
repeat_list

set()

In [8]:
summaries_dropped_index = summaries.drop(repeat_list).reset_index(drop=True)

In [29]:
# dropped = summaries.loc[list(repeat_list)]

In [35]:
# summaries.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/train_summary.csv", index=False)

In [50]:
# summaries_dropped_index = summaries

In [9]:
ages = []
for i, row in summaries_dropped_index.iterrows():
    pat = json.loads(row['patient_summary'])
    ages.append(pat['Subject Demographics']["Subject's age at visit"])

In [10]:
index = -1
incorrect_age_count = []
incorrect_cop_age_count = []

# Define regex pattern to extract and replace "a/an n-year-old"
remove_pattern = r"\b(\d{1,3})-year-old\s\b"

def remove_incorrect_age(text):
    global count
    global index
    global incorrect_cop_age
    index += 1
    # Process text line by line
    modified_text = []
    for sentence in text.split("."):
        match = re.search(remove_pattern, sentence)
        if match:  # Check if the sentence matches the pattern
            # print(sentence)
            age = match.group(1)  # Extract age
            # if "co-participant" in sentence.lower() or "spouse" in sentence.lower() or "child" in sentence.lower() or "daughter" in sentence.lower() or "friend" in sentence.lower() or "son" in sentence.lower() or "sibling" in sentence.lower() or "paid caregiver" in sentence.lower() or "relative" in sentence.lower():
            #     incorrect_cop_age_count.append((index, age))
                
            # else:   
            #     if int(age) != int(ages[index]):
            #         incorrect_age_count.append((index, age))
                    
            if int(age) != int(ages[index]):
                # print(text)
                incorrect_age_count.append((index, age, match))
            
    
    # return final_text

summaries_dropped_index["visit_summary"].apply(remove_incorrect_age)
print("Total sentences with incorrect co participant age:", len(incorrect_cop_age_count))
print("Total sentences with incorrect age:", len(incorrect_age_count))

Total sentences with incorrect co participant age: 0
Total sentences with incorrect age: 3


In [11]:
incorrect_age_count

[(31344, '40', <re.Match object; span=(34, 46), match='40-year-old '>),
 (31893, '35', <re.Match object; span=(80, 92), match='35-year-old '>),
 (35132, '35', <re.Match object; span=(55, 67), match='35-year-old '>)]

In [14]:
ages[35130]

83

In [17]:
print(summaries_dropped_index.iloc[35130]['visit_summary'])
# print(summaries_dropped_index.iloc[35130]['patient_summary'])

The patient is an 83-year-old right-handed, English-speaking woman who resides in a retirement community and lives independently. She has never been married and has 18 years of education. The primary reason for her visit to the Alzheimer's Disease Center is to participate in a research study, and she was referred by a professional contact. Her family history does not indicate any cognitive impairment in her parents, and no specific mutations were reported. She is currently taking five medications, including a lipid-lowering agent, an antihypertensive, an antiparkinson agent, and a beta-blocker. The medication listed as being used within two weeks of the visit is Mirabegron.

On physical examination, her resting heart rate is 72 beats per minute, her systolic and diastolic blood pressures are 111 mmHg and 68 mmHg, respectively, and her body mass index is 20.9. She is 60.8 inches tall and weighs 110 pounds. She does not use corrective lenses or a hearing aid, and her vision is functional

In [33]:
smoke_quit_age = []
for i, row in summaries_dropped_index.iterrows():
    pat = json.loads(row['patient_summary'])
    try:
        smoke_quit_age.append(pat['Subject Health History']["If the subject quit smoking, age at which he/she last smoked (i.e., quit) (range (7 - 110))"])
    except:
        smoke_quit_age.append(-1)

In [34]:
import re

# Define regex pattern to identify relevant sentences
pattern = r"\b\d{1,3}-year-old individual\b.*?\bsmoking\b|\bsmoking\b.*?\b\d{1,3}-year-old\b"

# Define regex pattern to extract and replace "a/an n-year-old"
remove_pattern = r"\b(\d{1,3})-year-old\s\b"

# Regex pattern to match "The patient quit smoking at the age of X"
quit_smoking_pattern = r"The patient quit smoking at the age of (\d+)"


index = 0
incorrect_smoking_age_count = []
invalid_smoking_age_count = []
quit_30_days_count = []

def clean_smoking_part(text):
    global count
    global index
    global invalid_smoking_age
    global quit_30_days_count
    
    modified_text = []
    
    for sentence in text.split("."):
        # sentence = sentence.strip()  # Remove leading/trailing spaces
        
        if "They quit smoking 30 days prior to the initial visit" in sentence:
            # sentence = sentence.replace("They quit smoking 30 days prior to the initial visit", "They did not smoke cigarettes for 30 days prior to the visit")
            quit_30_days_count.append(index)
        
        # Check for "The patient quit smoking at the age of X"
        quit_match = re.search(quit_smoking_pattern, sentence)
        if quit_match:
            age = int(quit_match.group(1))
            if int(age) > int(ages[index]):  # Skip this sentence if age < 60
                invalid_smoking_age_count.append(index)
                continue

        # Check for "n-year-old" + "smoking"
        match = re.search(remove_pattern, sentence)
        # if match and re.search(pattern, sentence):  
        if match and "smoking" in sentence.lower(): 
            age = match.group(1)  # Extract age
            if int(age) != int(ages[index]):
                # sentence = re.sub(remove_pattern, "", sentence).strip()   # Replace "n-year-old" with ""
                incorrect_smoking_age_count.append(index)
            
            if int(age) <= int(ages[index]) and (int(age) == int(smoke_quit_age[index]) and smoke_quit_age[index] != -1):
                # new_sentence = f"\n\nThe patient quit smoking at the age of {age}. "  # Create new sentence
                # modified_text.append(new_sentence + sentence)  # Add new sentence before original
                incorrect_smoking_age_count.append(index)
            # else:
            #     modified_text.append("\n\n" + sentence)
                # print(index)
                # raise ValueError
        # else:
        #     modified_text.append(sentence)  # Keep other sentences unchanged

    # Join sentences back into a full text
    # final_text = ".".join(modified_text)  # Removes any accidental empty sentences
    index += 1
    # return final_text

# Apply function to DataFrame column
summaries_dropped_index["visit_summary"].apply(clean_smoking_part)

print("Total sentences modified:", len(incorrect_smoking_age_count))
print("Invalid smoking age:", len(invalid_smoking_age_count))
print("Invalid quit_smoking_days:", len(quit_30_days_count))


Total sentences modified: 0
Invalid smoking age: 0
Invalid quit_smoking_days: 0
